In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (unit='s', utc=True)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) metric.csv summaries
# counts per (cmdb_id, kpi_name)
metric_group = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['cmdb_id', 'count'], ascending=[True, False])
)
# Limit to at most 20 rows for display
metric_group_head = metric_group.head(20)

# top 20 kpi_name overall by row count
metric_top20 = (
    metric_df['kpi_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'kpi_name', 'kpi_name': 'count'})
    .head(20)
)

# 2) trace.csv summaries
trace_group = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['cmdb_id', 'count'], ascending=[True, False])
)
trace_group_head = trace_group.head(20)

trace_top20 = (
    trace_df['trace_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'trace_name', 'trace_name': 'count'})
    .head(20)
)

# 3) log.csv summaries
log_group = (
    log_df
    .groupby(['cmdb_id', 'log_name'])
    .size()
    .reset_index(name='count')
    .sort_values(['cmdb_id', 'count'], ascending=[True, False])
)
log_group_head = log_group.head(20)

log_top20 = (
    log_df['log_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'log_name', 'log_name': 'count'})
    .head(20)
)

# 4) error_logs.csv summaries
# unique cmdb_id values and total row count per cmdb_id
error_counts = (
    error_df
    .groupby('cmdb_id')
    .agg(total_count=('message', 'size'), earliest_ts=('timestamp', 'min'))
    .reset_index()
    .sort_values('total_count', ascending=False)
)

# For each cmdb_id, collect up to 10 sample messages with their timestamps (ISO UTC)
def sample_messages_for_group(g):
    # sort by timestamp ascending
    s = g.sort_values('timestamp').head(10)
    # format as "2021-03-04T...Z - message"
    formatted = s.apply(lambda row: f"{row['timestamp'].isoformat()} - {row['message']}", axis=1).tolist()
    return formatted

if not error_df.empty:
    samples = (
        error_df
        .groupby('cmdb_id')
        .apply(sample_messages_for_group)
        .reset_index(name='sample_messages')
    )
    error_logs_summary = (
        error_counts.merge(samples, on='cmdb_id', how='left')
        .loc[:, ['cmdb_id', 'total_count', 'earliest_ts', 'sample_messages']]
    )
else:
    error_logs_summary = error_counts.assign(sample_messages=[[] for _ in range(len(error_counts))])

# Keep outputs compact as requested (limit rows)
metric_group_head, metric_top20, trace_group_head, trace_top20, log_group_head, log_top20, error_logs_summary.head(20)

```
Out[1]:
```
summary = (
    "Summary of telemetry files (concise):\n\n"
    "- metric.csv:\n"
    "  * Example services seen: ts-admin-basic-info-service, ts-admin-order-service, ts-admin-route-service, ts-admin-travel-service.\n"
    "  * Per-cmdb/kpi sample rows in the displayed output are ~25 each.\n"
    "  * Top metric KPIs by row count (overall top entries shown): cpu=1700, mem=1700, socket=1700, diskio=1675, "
    "latency-90=700, latency-50=700, workload=700, error=100.\n\n"
    "- trace.csv:\n"
    "  * Traces are rooted at 'root' and include many trace.to_*/trace.from_* features for downstream services.\n"
    "  * Per-cmdb trace_name groups in the sample are ~25 rows each.\n"
    "  * Top trace_names overall include: trace.from_root.row_count (340), trace.from_root.error_rate (340), "
    "trace.from_root.duration_p95 (340), trace.from_root.duration_mean (340), trace.from_ts-preserve-service.* (256 each), "
    "trace.from_ts-preserve-other-service.* (231 each), trace.to_ts-station-service.* (168), trace.to_ts-order-service.* (168).\n\n"
    "- log.csv:\n"
    "  * Typical per-cmdb log names: log.error_count and log.row_count (~24–25 rows per cmdb in sample).\n"
    "  * Top log names overall: log.error_count=1091, log.row_count=1091.\n\n"
    "- error_logs.csv:\n"
    "  * The error_logs file is empty in the displayed results; no sample error messages are available.\n\n"
    "Conclusion: Metric, trace, and log telemetry are broadly present. The most frequent metric KPIs are cpu, mem, and socket; "
    "logs are dominated by row_count and error_count. There are no raw error log messages to inspect from error_logs.csv."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  count
0   ts-admin-basic-info-service         cpu     25
1   ts-admin-basic-info-service      diskio     25
2   ts-admin-basic-info-service  latency-50     25
3   ts-admin-basic-info-service  latency-90     25
4   ts-admin-basic-info-service         mem     25
5   ts-admin-basic-info-service      socket     25
6   ts-admin-basic-info-service    workload     25
7        ts-admin-order-service         cpu     25
8        ts-admin-order-service      diskio     25
9        ts-admin-order-service         mem     25
10       ts-admin-order-service      socket     25
11       ts-admin-route-service         cpu     25
12       ts-admin-route-service      diskio     25
13       ts-admin-route-service         mem     25
14       ts-admin-route-service      socket     25
15      ts-admin-travel-service         cpu     25
16      ts-admin-travel-service      diskio     25
17      ts-admin-travel-service       error     25
18      ts-admin-travel-service  latency-50     25
19      ts-admin-travel-service  latency-90     25,         count  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4  latency-90    700
5  latency-50    700
6    workload    700
7       error    100,    cmdb_id                                         trace_name  count
0     root  trace.to_ts-admin-basic-info-service.duration_...     25
1     root  trace.to_ts-admin-basic-info-service.duration_p95     25
2     root    trace.to_ts-admin-basic-info-service.error_rate     25
3     root     trace.to_ts-admin-basic-info-service.row_count     25
4     root     trace.to_ts-admin-travel-service.duration_mean     25
5     root      trace.to_ts-admin-travel-service.duration_p95     25
6     root        trace.to_ts-admin-travel-service.error_rate     25
7     root         trace.to_ts-admin-travel-service.row_count     25
12    root             trace.to_ts-auth-service.duration_mean     25
13    root              trace.to_ts-auth-service.duration_p95     25
14    root                trace.to_ts-auth-service.error_rate     25
15    root                 trace.to_ts-auth-service.row_count     25
24    root             trace.to_ts-food-service.duration_mean     25
25    root              trace.to_ts-food-service.duration_p95     25
26    root                trace.to_ts-food-service.error_rate     25
27    root                 trace.to_ts-food-service.row_count     25
52    root           trace.to_ts-travel-service.duration_mean     25
53    root            trace.to_ts-travel-service.duration_p95     25
54    root              trace.to_ts-travel-service.error_rate     25
55    root               trace.to_ts-travel-service.row_count     25,                                                 count  count
0                           trace.from_root.row_count    340
1                          trace.from_root.error_rate    340
2                        trace.from_root.duration_p95    340
3                       trace.from_root.duration_mean    340
4            trace.from_ts-preserve-service.row_count    256
5           trace.from_ts-preserve-service.error_rate    256
6         trace.from_ts-preserve-service.duration_p95    256
7        trace.from_ts-preserve-service.duration_mean    256
8      trace.from_ts-preserve-other-service.row_count    231
9     trace.from_ts-preserve-other-service.error_rate    231
10  trace.from_ts-preserve-other-service.duration_p95    231
11  trace.from_ts-preserve-other-service.duration_...    231
12          trace.to_ts-station-service.duration_mean    168
13           trace.to_ts-station-service.duration_p95    168
14             trace.to_ts-station-service.error_rate    168
15              trace.to_ts-station-service.row_count    168
16                trace.to_ts-order-service.row_count    168
17               trace.to_ts-order-service.error_rate    168
18             trace.to_ts-order-service.duration_p95    168
19            trace.to_ts-order-service.duration_mean    168,                         cmdb_id         log_name  count
0   ts-admin-basic-info-service  log.error_count     25
1   ts-admin-basic-info-service    log.row_count     25
2       ts-admin-travel-service  log.error_count     25
3       ts-admin-travel-service    log.row_count     25
4            ts-assurance-mongo  log.error_count     24
5            ts-assurance-mongo    log.row_count     24
6          ts-assurance-service  log.error_count     24
7          ts-assurance-service    log.row_count     24
8                 ts-auth-mongo  log.error_count     24
9                 ts-auth-mongo    log.row_count     24
10              ts-auth-service  log.error_count     25
11              ts-auth-service    log.row_count     25
12             ts-basic-service  log.error_count     25
13             ts-basic-service    log.row_count     25
14              ts-config-mongo  log.error_count     24
15              ts-config-mongo    log.row_count     24
16             ts-consign-mongo  log.error_count     24
17             ts-consign-mongo    log.row_count     24
18       ts-consign-price-mongo  log.error_count     24
19       ts-consign-price-mongo    log.row_count     24,              count  count
0  log.error_count   1091
1    log.row_count   1091, Empty DataFrame
Columns: [cmdb_id, total_count, earliest_ts, sample_messages]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reload metric.csv and ensure timestamp parsed (reuse variable names if already in kernel)
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate components
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter metric rows to only these components for analysis
comp_metric = metric_df[metric_df['cmdb_id'].isin(components)].copy()

# Compute global P95 and total_points per (cmdb_id, kpi_name) using the full available series (requirement #9)
if not comp_metric.empty:
    global_aggs = (
        comp_metric
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .agg(total_points='count', p95=lambda x: x.quantile(0.95))
        .reset_index()
    )
else:
    global_aggs = pd.DataFrame(columns=['cmdb_id','kpi_name','total_points','p95'])

# Define incident window (UTC)
start = pd.to_datetime("2024-01-25 18:15:58", utc=True)
end = pd.to_datetime("2024-01-25 18:45:58", utc=True)

# Filter rows within the incident window
window_df = comp_metric[(comp_metric['timestamp'] >= start) & (comp_metric['timestamp'] <= end)].copy()

# Points in window per series
if not window_df.empty:
    points_in_window = (
        window_df
        .groupby(['cmdb_id','kpi_name'])
        .size()
        .reset_index(name='points_in_window')
    )
    # Merge p95 into window_df to detect anomalies (value >= p95)
    window_with_p95 = window_df.merge(global_aggs[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
    # Anomalies in window
    anomalies = (
        window_with_p95[window_with_p95['value'] >= window_with_p95['p95']]
        .groupby(['cmdb_id','kpi_name'])
        .agg(anomalies_in_window=('value','count'),
             earliest_anomaly_ts_utc=('timestamp','min'))
        .reset_index()
    )
    # max value in window
    max_in_window = (
        window_df
        .groupby(['cmdb_id','kpi_name'])['value']
        .max()
        .reset_index(name='max_value_in_window')
    )
else:
    points_in_window = pd.DataFrame(columns=['cmdb_id','kpi_name','points_in_window'])
    anomalies = pd.DataFrame(columns=['cmdb_id','kpi_name','anomalies_in_window','earliest_anomaly_ts_utc'])
    max_in_window = pd.DataFrame(columns=['cmdb_id','kpi_name','max_value_in_window'])

# Assemble final summary by merging global_aggs with window-derived summaries
summary = global_aggs.merge(points_in_window, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(anomalies, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(max_in_window, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0 and keep earliest_anomaly_ts_utc as NaT where missing
summary['points_in_window'] = summary['points_in_window'].fillna(0).astype(int)
summary['anomalies_in_window'] = summary['anomalies_in_window'].fillna(0).astype(int)
summary['earliest_anomaly_ts_utc'] = pd.to_datetime(summary['earliest_anomaly_ts_utc'], utc=True)
# max_value_in_window may remain NaN if no data in window

# Reorder columns as requested
summary = summary[[
    'cmdb_id','kpi_name','total_points','p95',
    'points_in_window','anomalies_in_window','earliest_anomaly_ts_utc','max_value_in_window'
]]

# Sort by anomalies_in_window desc, then by max_value_in_window desc
summary_sorted = summary.sort_values(by=['anomalies_in_window','max_value_in_window'], ascending=[False, False])

# Limit to top 20 rows for compact output
summary_top20 = summary_sorted.head(20)

summary_top20

```
Out[2]:
```
summary = (
    "Compact summary of results for the five candidate services (window: 2024-01-25 18:15:58–18:45:58 UTC):\n\n"
    "- General: For each reported (cmdb_id, kpi_name) the full series length (total_points) is 25 and the window contains all 25 points.\n"
    "- Anomalies: Every listed KPI shows anomalies_in_window = 2 (i.e., two points in the incident window exceed the global P95 for that series).\n\n"
    "- Top affected KPIs (by max value in window):\n"
    "  1) ts-travel-service — mem: p95 ~8.57e+08, max in window ~1.07e+09, earliest anomaly 2024-01-25T18:34:00Z.\n"
    "  2) ts-route-service — mem: p95 ~2.60e+08, max ~2.62e+08, earliest anomaly 2024-01-25T18:26:00Z.\n"
    "  3) ts-order-service — mem: p95 ~2.55e+08, max ~2.57e+08, earliest anomaly 2024-01-25T18:35:00Z.\n"
    "  4) ts-train-service — mem: p95 ~2.51e+08, max ~2.51e+08, earliest anomaly 2024-01-25T18:36:00Z.\n"
    "  5) ts-auth-service — mem: p95 ~2.38e+08, max ~2.38e+08, earliest anomaly 2024-01-25T18:41:00Z.\n"
    "  - Notable diskio spikes: ts-travel-service diskio max ~2.55e+07 (earliest anomaly 18:32), others smaller but present.\n"
    "  - CPU, socket, and workload KPIs also show 2 anomalies each but with lower peak magnitudes than memory.\n\n"
    "Conclusion: Memory-related metrics exhibit the largest spikes across multiple candidate services, with ts-travel-service.memory showing the highest peak in the incident window. All reported KPIs have two values exceeding their historical P95 within the window. "
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name  total_points           p95  points_in_window  anomalies_in_window   earliest_anomaly_ts_utc  max_value_in_window
32  ts-travel-service       mem            25  8.568397e+08                25                    2 2024-01-25 18:34:00+00:00         1.073741e+09
18   ts-route-service       mem            25  2.601706e+08                25                    2 2024-01-25 18:26:00+00:00         2.620486e+08
11   ts-order-service       mem            25  2.550244e+08                25                    2 2024-01-25 18:35:00+00:00         2.574030e+08
25   ts-train-service       mem            25  2.510520e+08                25                    2 2024-01-25 18:36:00+00:00         2.513951e+08
4     ts-auth-service       mem            25  2.378227e+08                25                    2 2024-01-25 18:41:00+00:00         2.381178e+08
29  ts-travel-service    diskio            25  2.427385e+07                25                    2 2024-01-25 18:32:00+00:00         2.551207e+07
15   ts-route-service    diskio            25  1.166411e+06                25                    2 2024-01-25 18:35:00+00:00         1.229830e+06
22   ts-train-service    diskio            25  7.236083e+04                25                    2 2024-01-25 18:18:00+00:00         1.042601e+05
1     ts-auth-service    diskio            25  6.423105e+04                25                    2 2024-01-25 18:21:00+00:00         9.085126e+04
8    ts-order-service    diskio            25  5.870961e+04                25                    2 2024-01-25 18:20:00+00:00         6.038174e+04
28  ts-travel-service       cpu            25  3.909539e+01                25                    2 2024-01-25 18:34:00+00:00         3.958029e+01
20   ts-route-service  workload            25  3.284142e+01                25                    2 2024-01-25 18:29:00+00:00         3.316452e+01
14   ts-route-service       cpu            25  1.676084e+01                25                    2 2024-01-25 18:18:00+00:00         2.753107e+01
33  ts-travel-service    socket            25  2.118000e+01                25                    2 2024-01-25 18:37:00+00:00         2.303333e+01
26   ts-train-service    socket            25  2.098000e+01                25                    2 2024-01-25 18:18:00+00:00         2.106667e+01
27   ts-train-service  workload            25  2.022756e+01                25                    2 2024-01-25 18:22:00+00:00         2.094200e+01
19   ts-route-service    socket            25  1.932000e+01                25                    2 2024-01-25 18:20:00+00:00         1.945000e+01
21   ts-train-service       cpu            25  1.321638e+01                25                    2 2024-01-25 18:18:00+00:00         1.845737e+01
0     ts-auth-service       cpu            25  1.631810e+01                25                    2 2024-01-25 18:18:00+00:00         1.769414e+01
5     ts-auth-service    socket            25  1.385333e+01                25                    2 2024-01-25 18:32:00+00:00         1.403333e+01```
```

In [ ]:
# In[3]:

import pandas as pd
import re
import numpy as np

# Load and parse trace.csv timestamps to UTC
trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services to match within trace_name
candidates = ["ts-travel-service","ts-route-service","ts-order-service","ts-train-service","ts-auth-service"]

# Build regex pattern and filter rows where trace_name contains any candidate substring
pattern = "|".join(re.escape(s) for s in candidates)
mask = trace_df['trace_name'].str.contains(pattern, na=False)
trace_focus = trace_df[mask].copy()

# Prepare empty outputs if no matching rows
if trace_focus.empty:
    A = pd.DataFrame(columns=['target_service','cmdb_id','trace_name','total_points','p95',
                              'points_in_window','anomalies_in_window','earliest_anomaly_ts_utc','max_value_in_window'])
    B = pd.DataFrame(columns=['target_service','total_trace_anomalies','earliest_anomaly_ts_utc_across_features',
                              'most_affected_trace_name','most_affected_max_value'])
else:
    # 1) Global aggregates (total_points, p95) per (cmdb_id, trace_name) using full series
    global_aggs = (
        trace_focus
        .groupby(['cmdb_id','trace_name'])['value']
        .agg(total_points='count', p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

    # Incident window (UTC)
    start = pd.to_datetime("2024-01-25 18:15:58", utc=True)
    end = pd.to_datetime("2024-01-25 18:45:58", utc=True)

    # 2) Rows in the incident window
    window_df = trace_focus[(trace_focus['timestamp'] >= start) & (trace_focus['timestamp'] <= end)].copy()

    # points_in_window per (cmdb_id, trace_name)
    points_in_window = (
        window_df
        .groupby(['cmdb_id','trace_name'])
        .size()
        .reset_index(name='points_in_window')
    )

    # Merge p95 into window rows to detect anomalies
    window_with_p95 = window_df.merge(global_aggs[['cmdb_id','trace_name','p95']], on=['cmdb_id','trace_name'], how='left')

    # anomalies_in_window and earliest anomaly timestamp
    if not window_with_p95.empty:
        anomalies = (
            window_with_p95[window_with_p95['value'] >= window_with_p95['p95']]
            .groupby(['cmdb_id','trace_name'])
            .agg(anomalies_in_window=('value','count'),
                 earliest_anomaly_ts_utc=('timestamp','min'))
            .reset_index()
        )
    else:
        anomalies = pd.DataFrame(columns=['cmdb_id','trace_name','anomalies_in_window','earliest_anomaly_ts_utc'])

    # max value in window per group
    if not window_df.empty:
        max_in_window = (
            window_df
            .groupby(['cmdb_id','trace_name'])['value']
            .max()
            .reset_index(name='max_value_in_window')
        )
    else:
        max_in_window = pd.DataFrame(columns=['cmdb_id','trace_name','max_value_in_window'])

    # Assemble per-(cmdb_id, trace_name) summary
    A = global_aggs.merge(points_in_window, on=['cmdb_id','trace_name'], how='left')
    A = A.merge(anomalies, on=['cmdb_id','trace_name'], how='left')
    A = A.merge(max_in_window, on=['cmdb_id','trace_name'], how='left')

    # Fill NA counts with 0 and normalize timestamp column
    A['points_in_window'] = A['points_in_window'].fillna(0).astype(int)
    A['anomalies_in_window'] = A['anomalies_in_window'].fillna(0).astype(int)
    A['earliest_anomaly_ts_utc'] = pd.to_datetime(A['earliest_anomaly_ts_utc'], utc=True)

    # Extract target_service label from trace_name
    def extract_target(trace_name):
        for s in candidates:
            if s in trace_name:
                return s
        return np.nan

    A['target_service'] = A['trace_name'].apply(extract_target)

    # Reorder columns as requested and sort by anomalies desc then max_value desc
    A = A[['target_service','cmdb_id','trace_name','total_points','p95',
           'points_in_window','anomalies_in_window','earliest_anomaly_ts_utc','max_value_in_window']]
    A = A.sort_values(by=['anomalies_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
    A = A.head(20)  # limit to top 20 rows

    # B) Per-target_service summary
    grp = A.groupby('target_service')

    B_counts = grp.agg(
        total_trace_anomalies=('anomalies_in_window','sum'),
        earliest_anomaly_ts_utc_across_features=('earliest_anomaly_ts_utc','min')
    ).reset_index()

    def most_affected(sub):
        # pick trace_name with highest max_value_in_window; treat NaN as -inf for sorting
        idx = sub['max_value_in_window'].fillna(-np.inf).idxmax()
        row = sub.loc[idx]
        return pd.Series({
            'most_affected_trace_name': row['trace_name'],
            'most_affected_max_value': row['max_value_in_window']
        })

    B_most = grp.apply(most_affected).reset_index()
    B = B_counts.merge(B_most, on='target_service', how='left')

# Return compact outputs A (per-feature) and B (per-target_service)
A, B

```
Out[3]:
```
Summary of trace analysis for candidate services (window 2024-01-25 18:15:58–18:45:58 UTC):

- What was measured:
  - We inspected trace features whose trace_name contains any of: ts-travel-service, ts-route-service, ts-order-service, ts-train-service, ts-auth-service.

- Per (cmdb_id, trace_name) findings (top 20 shown in results):
  - total_points (full series) = 25 for each listed trace_name.
  - global P95 computed from the full series = 0.0 for every feature.
  - points_in_window = 25 (the incident window contains all samples).
  - anomalies_in_window = 25 for each feature (every window point >= p95).
  - earliest anomaly timestamp (when any point >= p95) = 2024-01-25T18:18:00Z for all entries.
  - max_value_in_window = 0.0 for all entries.

- Per target-service aggregate:
  - ts-travel-service: total_trace_anomalies = 175, earliest anomaly 2024-01-25T18:18:00Z.
  - ts-train-service: total_trace_anomalies = 125, earliest anomaly 2024-01-25T18:18:00Z.
  - ts-route-service: total_trace_anomalies = 125, earliest anomaly 2024-01-25T18:18:00Z.
  - ts-auth-service: total_trace_anomalies = 75, earliest anomaly 2024-01-25T18:18:00Z.
  - For each target, the "most affected" trace_name reported has max_value_in_window = 0.0.

- Interpretation / note:
  - The global P95 for all inspected trace features is 0.0, and all observed values in the window are 0.0. Because we flagged anomalies as value >= P95, every point is counted as an "anomaly" (25/25). This is a statistical artefact: P95 = 0.0 with series of zeros causes every sample to meet the >= threshold, but there is no positive spike (max = 0.0).
  - Conclusion: there is no evidence of elevated trace error rates or latency spikes in these trace features during the window — values are zero throughout. The anomaly counts are not indicative of true service degradation given the zero-valued series.

The original code execution output of IPython Kernel is also provided below for reference:

(       target_service                  cmdb_id                               trace_name  total_points  p95  points_in_window  anomalies_in_window   earliest_anomaly_ts_utc  max_value_in_window
0     ts-auth-service                     root      trace.to_ts-auth-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
1   ts-travel-service                     root    trace.to_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
2   ts-travel-service  ts-admin-travel-service    trace.to_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
3     ts-auth-service          ts-auth-service    trace.from_ts-auth-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
4     ts-auth-service          ts-auth-service      trace.to_ts-auth-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
5    ts-route-service         ts-basic-service     trace.to_ts-route-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
6    ts-train-service         ts-basic-service     trace.to_ts-train-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
7   ts-travel-service          ts-food-service    trace.to_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
8    ts-route-service         ts-route-service   trace.from_ts-route-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
9   ts-travel-service         ts-route-service  trace.from_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
10   ts-route-service         ts-route-service     trace.to_ts-route-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
11   ts-train-service         ts-train-service   trace.from_ts-train-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
12  ts-travel-service         ts-train-service  trace.from_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
13   ts-train-service         ts-train-service     trace.to_ts-train-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
14  ts-travel-service        ts-travel-service  trace.from_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
15   ts-route-service        ts-travel-service     trace.to_ts-route-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
16   ts-train-service        ts-travel-service     trace.to_ts-train-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
17  ts-travel-service        ts-travel-service    trace.to_ts-travel-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
18   ts-route-service       ts-travel2-service     trace.to_ts-route-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0
19   ts-train-service       ts-travel2-service     trace.to_ts-train-service.error_rate            25  0.0                25                   25 2024-01-25 18:18:00+00:00                  0.0,       target_service  total_trace_anomalies earliest_anomaly_ts_utc_across_features               most_affected_trace_name  most_affected_max_value
0    ts-auth-service                     75               2024-01-25 18:18:00+00:00    trace.to_ts-auth-service.error_rate                      0.0
1   ts-route-service                    125               2024-01-25 18:18:00+00:00   trace.to_ts-route-service.error_rate                      0.0
2   ts-train-service                    125               2024-01-25 18:18:00+00:00   trace.to_ts-train-service.error_rate                      0.0
3  ts-travel-service                    175               2024-01-25 18:18:00+00:00  trace.to_ts-travel-service.error_rate                      0.0)```
```

In [ ]:
# In[4]:

import pandas as pd

# Re-load log and error log files (timestamps will be parsed to UTC)
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter logs to only candidate services
log_focus = log_df[log_df['cmdb_id'].isin(candidates)].copy()

# If no matching rows, prepare empty summary
if log_focus.empty:
    log_summary_top20 = pd.DataFrame(columns=[
        'cmdb_id','log_name','total_points','p95',
        'points_in_window','anomalies_in_window','earliest_anomaly_ts_utc','max_value_in_window'
    ])
else:
    # 1) Global P95 and total_points per (cmdb_id, log_name) using the full series
    global_aggs = (
        log_focus
        .groupby(['cmdb_id','log_name'])['value']
        .agg(total_points='count', p95=lambda x: x.quantile(0.95))
        .reset_index()
    )

    # Incident window (UTC)
    start = pd.to_datetime("2024-01-25 18:15:58", utc=True)
    end = pd.to_datetime("2024-01-25 18:45:58", utc=True)

    # Rows within the incident window
    window_df = log_focus[(log_focus['timestamp'] >= start) & (log_focus['timestamp'] <= end)].copy()

    # points_in_window per (cmdb_id, log_name)
    if not window_df.empty:
        points_in_window = (
            window_df
            .groupby(['cmdb_id','log_name'])
            .size()
            .reset_index(name='points_in_window')
        )
        # merge p95 for anomaly detection
        window_with_p95 = window_df.merge(global_aggs[['cmdb_id','log_name','p95']], on=['cmdb_id','log_name'], how='left')

        anomalies = (
            window_with_p95[window_with_p95['value'] >= window_with_p95['p95']]
            .groupby(['cmdb_id','log_name'])
            .agg(anomalies_in_window=('value','count'),
                 earliest_anomaly_ts_utc=('timestamp','min'))
            .reset_index()
        )

        max_in_window = (
            window_df
            .groupby(['cmdb_id','log_name'])['value']
            .max()
            .reset_index(name='max_value_in_window')
        )
    else:
        points_in_window = pd.DataFrame(columns=['cmdb_id','log_name','points_in_window'])
        anomalies = pd.DataFrame(columns=['cmdb_id','log_name','anomalies_in_window','earliest_anomaly_ts_utc'])
        max_in_window = pd.DataFrame(columns=['cmdb_id','log_name','max_value_in_window'])

    # Assemble final summary
    log_summary = global_aggs.merge(points_in_window, on=['cmdb_id','log_name'], how='left')
    log_summary = log_summary.merge(anomalies, on=['cmdb_id','log_name'], how='left')
    log_summary = log_summary.merge(max_in_window, on=['cmdb_id','log_name'], how='left')

    # Fill NA counts with 0 and normalize timestamp
    log_summary['points_in_window'] = log_summary['points_in_window'].fillna(0).astype(int)
    log_summary['anomalies_in_window'] = log_summary['anomalies_in_window'].fillna(0).astype(int)
    log_summary['earliest_anomaly_ts_utc'] = pd.to_datetime(log_summary['earliest_anomaly_ts_utc'], utc=True)

    # Reorder and sort as requested
    log_summary = log_summary[[
        'cmdb_id','log_name','total_points','p95',
        'points_in_window','anomalies_in_window','earliest_anomaly_ts_utc','max_value_in_window'
    ]]

    log_summary_top20 = log_summary.sort_values(
        by=['anomalies_in_window','max_value_in_window'],
        ascending=[False, False]
    ).head(20).reset_index(drop=True)

# Error logs overview
total_error_count = len(error_df)
# Incident window filtering for error logs
start = pd.to_datetime("2024-01-25 18:15:58", utc=True)
end = pd.to_datetime("2024-01-25 18:45:58", utc=True)
error_window = error_df[(error_df['timestamp'] >= start) & (error_df['timestamp'] <= end)].copy()
error_count_in_window = len(error_window)

# Up to 10 sample messages with timestamps if any in-window
if error_count_in_window > 0:
    error_samples = error_window.sort_values('timestamp').head(10)[['timestamp','cmdb_id','message']].reset_index(drop=True)
else:
    # empty DataFrame with desired columns
    error_samples = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Compact summary DataFrame for error overview
error_overview = pd.DataFrame([{
    'total_error_count': total_error_count,
    'error_count_in_window': error_count_in_window
}])

# Return compact outputs
log_summary_top20, error_overview, error_samples

```
Out[4]:
```
Summary of log and error-log analysis for candidate services (window 2024-01-25 18:15:58–18:45:58 UTC):

- error_logs.csv:
  - No error log rows at all (total_error_count = 0). No error messages in the incident window.

- log.csv (candidate services: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service):
  - log.error_count:
    - For ts-auth, ts-order, ts-route, ts-travel: total_points ≈ 24–25, p95 = 0.00, points_in_window = all samples, anomalies_in_window = all samples (24–25).
    - Interpretation: series are zero-valued so p95 = 0 and every point meets value >= p95. This is a statistical artifact and does not indicate real error spikes.
  - log.row_count (meaningful activity spikes):
    - ts-travel-service: total_points = 25, global p95 ≈ 2043.6, points_in_window = 25, anomalies_in_window = 2, earliest anomaly 2024-01-25T18:22:00Z, max in window = 2139.
    - ts-route-service: total_points = 25, p95 ≈ 994.8, points_in_window = 25, anomalies_in_window = 2, earliest anomaly 2024-01-25T18:27:00Z, max = 1005.
    - ts-order-service: total_points = 24, p95 ≈ 469.55, points_in_window = 24, anomalies_in_window = 2, earliest anomaly 2024-01-25T18:22:00Z, max = 484.
    - ts-auth-service: total_points = 25, p95 ≈ 326.40, points_in_window = 25, anomalies_in_window = 2, earliest anomaly 2024-01-25T18:27:00Z, max = 328.
    - Interpretation: these services show two samples in the incident window that exceed their historical P95 for log.row_count — indicating short spikes in request/log volume (ts-travel-service has the largest spike).

Overall conclusion:
- No error log messages to inspect.
- The meaningful signals are spikes in log.row_count for ts-travel-service (largest), ts-route-service, ts-order-service, and ts-auth-service (each with two points exceeding their global P95 in the window). The many "anomalies" for log.error_count are artifacts caused by p95 = 0 for zero-valued series and should be ignored.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  total_points      p95  points_in_window  anomalies_in_window   earliest_anomaly_ts_utc  max_value_in_window
0    ts-auth-service  log.error_count            25     0.00                25                   25 2024-01-25 18:18:00+00:00                    0
1   ts-route-service  log.error_count            25     0.00                25                   25 2024-01-25 18:18:00+00:00                    0
2  ts-travel-service  log.error_count            25     0.00                25                   25 2024-01-25 18:18:00+00:00                    0
3   ts-order-service  log.error_count            24     0.00                24                   24 2024-01-25 18:19:00+00:00                    0
4  ts-travel-service    log.row_count            25  2043.60                25                    2 2024-01-25 18:22:00+00:00                 2139
5   ts-route-service    log.row_count            25   994.80                25                    2 2024-01-25 18:27:00+00:00                 1005
6   ts-order-service    log.row_count            24   469.55                24                    2 2024-01-25 18:22:00+00:00                  484
7    ts-auth-service    log.row_count            25   326.40                25                    2 2024-01-25 18:27:00+00:00                  328,    total_error_count  error_count_in_window
0                  0                      0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```